# Stock Predictor - Reliance (RELIANCE.NS)

Predicting next-day movement of Reliance Industries using technical indicators + LSTMs.

Two tasks:
1. **Classification** - will tomorrow's close be higher than today's?
2. **Regression** - what will tomorrow's close be? (predicted as a return, then turned back into a price)

Every model gets compared against a dumb baseline, because on daily stock data
"looks good on a chart" and "actually beats guessing" are very different things.

*Just a learning project, not financial advice.*

In [ ]:
# on colab only yfinance / keras-tuner might be missing
!pip install -q yfinance keras-tuner

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error)

warnings.filterwarnings('ignore')
%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TICKER = 'RELIANCE.NS'
START_DATE = '2008-01-01'
LOOKBACK = 60                    # days of history the LSTM sees
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
RUN_TUNER = False                # hyperband search takes ~50 min on colab

## 1. Data

In [ ]:
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True, progress=False)

# yfinance gives MultiIndex columns (Price, Ticker) even for one ticker, and the order is
# Close, High, Low, Open, Volume -> flatten by name, renaming by position swaps Open/Close
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.index = pd.to_datetime(df.index)

print(f'{len(df)} rows, {df.index.min().date()} -> {df.index.max().date()}')
df.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df['Close'], color='tab:blue', label='Close')
plt.title(f'{TICKER} closing price')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Data quality checks

In [ ]:
print('missing values:')
print(df.isnull().sum())

zero_vol = df[df['Volume'] == 0]
bad_ohlc = df[(df['High'] < df[['Open', 'Close']].max(axis=1)) |
              (df['Low'] > df[['Open', 'Close']].min(axis=1)) |
              (df['High'] < df['Low'])]
flat_days = df[df['Open'] == df['Close']]

print(f'\nzero volume days:     {len(zero_vol)}')
print(f'impossible OHLC rows: {len(bad_ohlc)}')
print(f'open == close days:   {len(flat_days)} (this is fine, just curious)')

if len(bad_ohlc):
    display(bad_ohlc.head())

# zero volume rows are usually holidays / bad prints and they break MFI + OBV
df = df.drop(index=zero_vol.index.union(bad_ohlc.index)).dropna()
print(f'\nrows after cleaning: {len(df)}')

## 2. Features

I used to do this with `pandas_ta`, but its column names changed between versions
(Bollinger Bands and ATR were silently getting skipped), so now I compute everything myself.

Rules I'm following:
- a feature on day *t* can only use data up to the close of day *t*
- no raw price-level features (raw OBV, raw moving averages...). Reliance trades way higher
  in the test period than in training, so those end up outside the scaler's range.
  Everything is a ratio / distance / oscillator instead.

In [ ]:
def wilder(s, n):
    return s.ewm(alpha=1 / n, adjust=False).mean()


def true_range(h, l, c):
    prev_close = c.shift(1)
    return pd.concat([h - l, (h - prev_close).abs(), (l - prev_close).abs()], axis=1).max(axis=1)


def alma(s, n=9, offset=0.85, sigma=6):
    m = offset * (n - 1)
    w = np.exp(-((np.arange(n) - m) ** 2) / (2 * (n / sigma) ** 2))
    w /= w.sum()
    return s.rolling(n).apply(lambda x: np.dot(x, w), raw=True)


def zero_lag_ema(s, n):
    ema1 = s.ewm(span=n, adjust=False).mean()
    ema2 = ema1.ewm(span=n, adjust=False).mean()
    return 2 * ema1 - ema2


def build_features(df):
    o, h, l, c, v = df['Open'], df['High'], df['Low'], df['Close'], df['Volume']
    f = pd.DataFrame(index=df.index)

    # basic price action
    for n in (1, 5, 10, 20):
        f[f'ret_{n}'] = c.pct_change(n)
    f['gap'] = o / c.shift(1) - 1
    f['body'] = (c - o) / c
    f['close_vs_sma50'] = c / c.rolling(50).mean() - 1
    f['vol_ratio'] = v / v.rolling(20).mean()

    # ichimoku - spans are shifted forward 26 days like on the chart, so they only use past data.
    # skipping the chikou span on purpose: it's close shifted BACK 26 days = future data
    tenkan = (h.rolling(9).max() + l.rolling(9).min()) / 2
    kijun = (h.rolling(26).max() + l.rolling(26).min()) / 2
    span_a = ((tenkan + kijun) / 2).shift(26)
    span_b = ((h.rolling(52).max() + l.rolling(52).min()) / 2).shift(26)
    cloud_top, cloud_bot = np.maximum(span_a, span_b), np.minimum(span_a, span_b)
    f['price_vs_cloud'] = np.select([c > cloud_top, c < cloud_bot], [1, -1], 0)
    f['cloud_dist'] = (c - (span_a + span_b) / 2) / c
    f['tenkan_vs_kijun'] = np.where(tenkan > kijun, 1, -1)
    f['tk_dist'] = (tenkan - kijun) / c

    # alma crossover
    a9, a21 = alma(c, 9), alma(c, 21)
    f['alma_signal'] = np.where(a9 > a21, 1, -1)
    f['alma_dist'] = (a9 - a21) / c

    # vortex
    tr = true_range(h, l, c)
    tr14 = tr.rolling(14).sum()
    vi_plus = (h - l.shift(1)).abs().rolling(14).sum() / tr14
    vi_minus = (l - h.shift(1)).abs().rolling(14).sum() / tr14
    f['vi_diff'] = vi_plus - vi_minus
    f['vi_signal'] = np.where(vi_plus > vi_minus, 1, -1)

    # adx / directional movement
    atr = wilder(tr, 14)
    up, down = h.diff(), -l.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=df.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=df.index)
    plus_di = 100 * wilder(plus_dm, 14) / atr
    minus_di = 100 * wilder(minus_dm, 14) / atr
    adx = wilder(100 * (plus_di - minus_di).abs() / (plus_di + minus_di), 14)
    f['adx'] = adx
    f['di_diff'] = plus_di - minus_di
    f['adx_trending'] = (adx > 25).astype(int)

    # tsi (25, 13, 13)
    m = c.diff()
    num = m.ewm(span=25, adjust=False).mean().ewm(span=13, adjust=False).mean()
    den = m.abs().ewm(span=25, adjust=False).mean().ewm(span=13, adjust=False).mean()
    tsi = 100 * num / den
    tsi_sig = tsi.ewm(span=13, adjust=False).mean()
    f['tsi'] = tsi
    f['tsi_cross'] = np.where(tsi > tsi_sig, 1, -1)

    # zero lag macd (12, 26, 9)
    zl = zero_lag_ema(c, 12) - zero_lag_ema(c, 26)
    zl_sig = zl.ewm(span=9, adjust=False).mean()
    f['zlmacd_hist'] = (zl - zl_sig) / c
    f['zlmacd_signal'] = np.where(zl > zl_sig, 1, -1)

    # stochastic (14, 3, 3)
    ll, hh = l.rolling(14).min(), h.rolling(14).max()
    k = (100 * (c - ll) / (hh - ll)).rolling(3).mean()
    d = k.rolling(3).mean()
    f['stoch_k'] = k
    f['stoch_kd'] = k - d
    f['stoch_signal'] = np.select([k > 80, k < 20], [-1, 1], 0)

    # bollinger bands (20, 2)
    mid = c.rolling(20).mean()
    sd = c.rolling(20).std(ddof=0)
    bb_up, bb_lo = mid + 2 * sd, mid - 2 * sd
    f['bb_width'] = (bb_up - bb_lo) / mid
    f['bb_position'] = (c - bb_lo) / (bb_up - bb_lo)

    # atr as % of price
    f['atr_pct'] = 100 * atr / c

    # mfi
    tp = (h + l + c) / 3
    mf = tp * v
    pos = mf.where(tp > tp.shift(1), 0.0).rolling(14).sum()
    neg = mf.where(tp < tp.shift(1), 0.0).rolling(14).sum()
    mfi = 100 - 100 / (1 + pos / neg)
    f['mfi'] = mfi
    f['mfi_signal'] = np.select([mfi > 80, mfi < 20], [-1, 1], 0)

    # obv - raw obv is a running total, so use its 5 day change relative to volume
    obv = (np.sign(c.diff()).fillna(0) * v).cumsum()
    f['obv_slope'] = obv.diff(5) / v.rolling(5).sum()
    f['obv_trend'] = np.sign(obv.diff(5))

    # ttm squeeze: bollinger bands fully inside keltner channels
    kc_mid = c.ewm(span=20, adjust=False).mean()
    atr20 = tr.rolling(20).mean()
    f['squeeze_on'] = ((bb_up < kc_mid + 1.5 * atr20) & (bb_lo > kc_mid - 1.5 * atr20)).astype(int)

    # fair value gap: 1 = bullish gap (today's low above the high 2 days ago), -1 = bearish
    f['fvg'] = np.select([l > h.shift(2), h < l.shift(2)], [1, -1], 0)

    return f

In [ ]:
features = build_features(df)

# label for day t = does tomorrow close higher than today
next_close = df['Close'].shift(-1)
df['target'] = (next_close > df['Close']).astype(int)
df['next_ret'] = np.log(next_close / df['Close'])

data = features.join(df[['Close', 'target', 'next_ret']])
data = data.replace([np.inf, -np.inf], np.nan).dropna()   # also drops the last day (no tomorrow yet)
feature_cols = list(features.columns)

print(f'{len(data)} rows, {len(feature_cols)} features')
print(f"up days: {data['target'].mean():.1%}")

In [ ]:
# lookahead check: if I chop off the second half of the data, the features for the
# first half shouldn't change at all. if any feature peeks at the future this fails
cut = df.index[len(df) // 2]
pd.testing.assert_frame_equal(build_features(df.loc[:cut]), features.loc[:cut], check_dtype=False)
print('no lookahead in features')

## 3. Train / val / test split + feature selection

In [ ]:
n = len(data)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

print(f'train: {data.index[0].date()} -> {data.index[train_end - 1].date()}  ({train_end} days)')
print(f'val:   {data.index[train_end].date()} -> {data.index[val_end - 1].date()}  ({val_end - train_end} days)')
print(f'test:  {data.index[val_end].date()} -> {data.index[-1].date()}  ({n - val_end} days)')

# scaler only ever sees training data
scaler = MinMaxScaler()
scaler.fit(data[feature_cols].iloc[:train_end])
X_all = pd.DataFrame(scaler.transform(data[feature_cols]), index=data.index, columns=feature_cols)
y_all = data['target']

X_train, X_val, X_test = X_all.iloc[:train_end], X_all.iloc[train_end:val_end], X_all.iloc[val_end:]
y_train, y_val, y_test = y_all.iloc[:train_end], y_all.iloc[train_end:val_end], y_all.iloc[val_end:]

In [ ]:
# step 1: drop one of each pair of highly correlated features (train data only)
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
to_drop = [col for col in upper.columns if (upper[col] > 0.9).any()]
kept = [col for col in feature_cols if col not in to_drop]
print(f'correlation filter: {len(feature_cols)} -> {len(kept)} features')
print('dropped:', to_drop)

# step 2: random forest importance
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                            class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train[kept], y_train)

majority = int(y_train.mean() >= 0.5)
rf_val_proba = rf.predict_proba(X_val[kept])[:, 1]
print(f'\nval accuracy, always guess majority class: {(y_val == majority).mean():.4f}')
print(f'val accuracy, random forest:               {accuracy_score(y_val, (rf_val_proba > 0.5).astype(int)):.4f}')
print(f'val roc-auc, random forest:                {roc_auc_score(y_val, rf_val_proba):.4f}')

importance = pd.Series(rf.feature_importances_, index=kept).sort_values(ascending=False)
selected = importance[importance > 0.01].index.tolist()
print(f'\nimportance filter: {len(kept)} -> {len(selected)} features')

plt.figure(figsize=(8, 7))
sns.barplot(x=importance.values, y=importance.index, color='tab:blue')
plt.axvline(0.01, color='red', ls='--', lw=1)
plt.title('Random forest feature importance (train)')
plt.tight_layout()
plt.show()

## 4. LSTM sequences

Each sample is a window of the last `LOOKBACK` days ending on day *t*, and its label is day *t*'s target
(tomorrow up or down). Val/test windows are allowed to reach back into the previous split -
that's just past data, and it means I don't throw away the first 60 days of val and test.

In [ ]:
def make_sequences(X, y, lookback, start, end):
    Xv, yv = X.values, y.values
    days = np.arange(max(start, lookback - 1), end)
    xs = np.stack([Xv[t - lookback + 1:t + 1] for t in days]).astype('float32')
    return xs, yv[days], days


X_sel = X_all[selected]
X_tr, y_tr, tr_days = make_sequences(X_sel, y_all, LOOKBACK, 0, train_end)
X_va, y_va, va_days = make_sequences(X_sel, y_all, LOOKBACK, train_end, val_end)
X_te, y_te, te_days = make_sequences(X_sel, y_all, LOOKBACK, val_end, n)
n_feat = X_tr.shape[2]

print('train:', X_tr.shape, ' val:', X_va.shape, ' test:', X_te.shape)

## 5. LSTM classifier (up / down)

In [ ]:
import tensorflow as tf
from tensorflow.keras import Input, Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(SEED)


def lstm_model(n_features, units=50, dropout=0.2, lr=1e-3, task='clf'):
    model = Sequential([
        Input(shape=(LOOKBACK, n_features)),
        LSTM(units, return_sequences=True),
        Dropout(dropout),
        LSTM(units),
        Dropout(dropout),
        Dense(1, activation='sigmoid' if task == 'clf' else None),
    ])
    if task == 'clf':
        model.compile(optimizer=Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    else:
        model.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return model


def early_stop():
    # new callback each time so state doesn't carry over between models
    return EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


def plot_history(hist, title):
    plt.figure(figsize=(8, 4))
    plt.plot(hist.history['loss'], label='train')
    plt.plot(hist.history['val_loss'], label='val')
    plt.title(title)
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
clf = lstm_model(n_feat, task='clf')
hist_clf = clf.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                   epochs=100, batch_size=32, callbacks=[early_stop()], verbose=2)
plot_history(hist_clf, 'LSTM classifier loss')

In [ ]:
clf_proba = clf.predict(X_te, verbose=0).ravel()
clf_pred = (clf_proba > 0.5).astype(int)
rf_test_proba = rf.predict_proba(X_test[kept])[:, 1]

clf_results = pd.DataFrame({
    'accuracy': [(y_te == majority).mean(),
                 accuracy_score(y_test, (rf_test_proba > 0.5).astype(int)),
                 accuracy_score(y_te, clf_pred)],
    'roc_auc': [0.5, roc_auc_score(y_test, rf_test_proba), roc_auc_score(y_te, clf_proba)],
}, index=['majority class', 'random forest', 'LSTM'])
print('test set:')
display(clf_results.round(4))

print(classification_report(y_te, clf_pred, target_names=['Down', 'Up'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_te, clf_pred, display_labels=['Down', 'Up'], cmap='Blues')
plt.title('LSTM classifier - test')
plt.show()

## 6. LSTM regression (next-day price)

First version of this predicted the scaled price directly. Two problems with that:
- the test period has prices higher than anything in training, so the scaled target goes past 1
  and the model can't really get there
- a model that just outputs "yesterday's price" already gets a tiny error and a nice looking plot

So now the model predicts tomorrow's **log return**, I convert that back to a price
(`close_today * exp(pred)`), and compare against the naive guess *tomorrow = today*.

In [ ]:
# return scaler fitted on train only
ret_scaler = StandardScaler()
ret_scaler.fit(data[['next_ret']].iloc[:train_end])
r_all = pd.Series(ret_scaler.transform(data[['next_ret']]).ravel(), index=data.index)

r_tr, r_va, r_te = r_all.values[tr_days], r_all.values[va_days], r_all.values[te_days]

close_today = data['Close'].values[te_days]
actual_next = close_today * np.exp(data['next_ret'].values[te_days])
test_dates = data.index[te_days]


def evaluate_regression(model, name):
    pred_ret = ret_scaler.inverse_transform(model.predict(X_te, verbose=0)).ravel()
    pred_next = close_today * np.exp(pred_ret)
    actual_ret = data['next_ret'].values[te_days]

    rows = {
        'naive (tomorrow = today)': close_today,
        name: pred_next,
    }
    out = pd.DataFrame({
        'MAE': [mean_absolute_error(actual_next, p) for p in rows.values()],
        'RMSE': [np.sqrt(mean_squared_error(actual_next, p)) for p in rows.values()],
    }, index=rows.keys())
    out['direction_acc'] = [np.nan, (np.sign(pred_ret) == np.sign(actual_ret)).mean()]
    display(out.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1]})
    axes[0].plot(test_dates, actual_next, label='actual', color='tab:blue')
    axes[0].plot(test_dates, pred_next, label='predicted', color='tab:red', alpha=0.7)
    axes[0].set_title(f'{TICKER} next-day close - {name}')
    axes[0].set_ylabel('Price (INR)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # zoomed in returns, this is where you can actually see if it's learning anything
    axes[1].plot(test_dates[-90:], actual_ret[-90:], label='actual return', color='tab:blue')
    axes[1].plot(test_dates[-90:], pred_ret[-90:], label='predicted return', color='tab:red')
    axes[1].axhline(0, color='black', lw=0.8)
    axes[1].set_title('last 90 days - log returns')
    axes[1].legend()
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    return out


reg = lstm_model(n_feat, task='reg')
hist_reg = reg.fit(X_tr, r_tr, validation_data=(X_va, r_va),
                   epochs=100, batch_size=32, callbacks=[early_stop()], verbose=2)
plot_history(hist_reg, 'LSTM regression loss')
reg_results = evaluate_regression(reg, 'LSTM')

### Hyperparameter tuning (Keras Tuner - Hyperband)

Set `RUN_TUNER = True` at the top to re-run the search. Otherwise it uses the values from my last run.

In [ ]:
if RUN_TUNER:
    import keras_tuner as kt

    def build_tuned(hp):
        return lstm_model(n_feat,
                          units=hp.Int('units', 32, 128, step=32),
                          dropout=hp.Float('dropout', 0.1, 0.4, step=0.1),
                          lr=hp.Choice('lr', [1e-2, 1e-3, 1e-4]),
                          task='reg')

    tuner = kt.Hyperband(build_tuned, objective='val_loss', max_epochs=50, factor=3,
                         directory='tuner_logs', project_name='stock_lstm',
                         overwrite=True, seed=SEED)
    tuner.search(X_tr, r_tr, validation_data=(X_va, r_va), callbacks=[early_stop()], verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0].values
else:
    # from my earlier hyperband run (61 trials). that run was on the old price target,
    # so re-run the search if you want properly tuned values
    best_hp = {'units': 96, 'dropout': 0.2, 'lr': 1e-3}

print('using:', best_hp)

In [ ]:
reg_tuned = lstm_model(n_feat, units=best_hp['units'], dropout=best_hp['dropout'],
                       lr=best_hp['lr'], task='reg')
hist_tuned = reg_tuned.fit(X_tr, r_tr, validation_data=(X_va, r_va),
                           epochs=100, batch_size=32, callbacks=[early_stop()], verbose=2)
plot_history(hist_tuned, 'Tuned LSTM regression loss')
tuned_results = evaluate_regression(reg_tuned, 'tuned LSTM')

## 7. Notes

- If the LSTM accuracy lands around the majority-class baseline, that's not a bug.
  Daily returns are close to a random walk, and most of the "70% accuracy" stock notebooks out there
  have some kind of lookahead in them (mine did too - see the README).
- Same for regression: beating *tomorrow = today* on MAE is the real test, not how close the two lines look.
- Ideas to try next: walk-forward validation instead of one split, adding market context (NIFTY 50, INR/USD),
  predicting multi-day returns where there's a bit more signal, and a simple backtest with transaction costs.